# 01: Download the data and check its structure

**What this notebook does**
1. Gets the tools ready (DuckDB and pandas).
2. Downloads the two data files into Colab: **reviews** and **products (meta)**.
3. Shows the **structure** of each file (column names and types) without loading the whole file into memory.

**Before you run it:** read [docs/04-colab-setup.md](../docs/04-colab-setup.md). The files are huge; this notebook never loads them fully, on purpose.

**Time:** the downloads take a few minutes.

## Step 1: Get the tools ready

In [ ]:
# "!" means: run this as a computer command, not as Python.
# "pip install" adds a Python library. Colab usually has DuckDB already, so this may just say "already satisfied".
!pip install duckdb

# DuckDB: reads big files straight from disk, without loading them all into memory.
import duckdb

# pandas: works with tables (called DataFrames) once the data is small enough.
import pandas as pd

# gc ("garbage collector"): frees up memory after we delete big tables we no longer need.
import gc

# Cap DuckDB's memory at 6 GB, so it can't use up all of Colab's ~12.7 GB.
# Note: this cap covers DuckDB only, not pandas tables.
duckdb.sql("SET memory_limit = '6GB'")

## Step 2: Save the download addresses

Each address goes into a variable (a named box), so the links are easy to read and reuse.
These are the addresses listed on the official page: https://amazon-reviews-2023.github.io

In [ ]:
# Address of the REVIEW file (one row = one customer review).
review_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Automotive.jsonl.gz"

# Address of the META file (one row = one product and its details).
meta_url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Automotive.jsonl.gz"

## Step 3: Download both files

`wget` is a small tool that downloads a file from a web address.
`{review_url}` tells Colab: put the address saved in `review_url` here.

**Run this cell only once per session.** If you run it again, `wget` doesn't overwrite. It saves a second copy ending in `.1`.

**Don't write `review_file = !wget ...`.** That saves wget's progress messages into the variable, not the file name, and breaks the later steps.

In [ ]:
# Download the review file into Colab (not your laptop).
!wget {review_url}

# Download the meta file into Colab.
!wget {meta_url}

## Step 4: Save the file names and check the downloads

In [ ]:
# The plain file names that wget saved. Later steps use these.
review_file = "Automotive.jsonl.gz"
meta_file = "meta_Automotive.jsonl.gz"

# Print the names, to check they are clean file names.
print(review_file)
print(meta_file)

In [ ]:
# "ls" lists the files in this folder. "-lh" shows sizes in easy units, like MB or GB.
# Check: both files are there, and neither is tiny (a few KB means the download failed).
!ls -lh

## Step 5: Look at the structure of each file

`DESCRIBE` asks DuckDB for the **column names and types** only, not the data.
`read_json_auto(...)` tells DuckDB to read the JSON Lines file and work out the columns itself. It reads the zipped `.gz` file directly.

So this is quick, and it doesn't load the files into memory.

In [ ]:
# Build the question for DuckDB: "describe every column of the review file".
review_structure_query = f"DESCRIBE SELECT * FROM read_json_auto('{review_file}')"

# Send the question to DuckDB and keep the answer.
review_structure = duckdb.sql(review_structure_query)

# Show the answer: one row per column, with its type.
print(review_structure)

In [ ]:
# Same thing for the meta (product) file.
meta_structure_query = f"DESCRIBE SELECT * FROM read_json_auto('{meta_file}')"

# Send the question to DuckDB and keep the answer.
meta_structure = duckdb.sql(meta_structure_query)

# Show the answer.
print(meta_structure)

## What we found

**Review file: 10 columns**
`rating`, `title`, `text`, `images`, `asin`, `parent_asin`, `user_id`, `timestamp`, `helpful_vote`, `verified_purchase`

**Meta file: 16 columns**
`main_category`, `title`, `average_rating`, `rating_number`, `features`, `description`, `price`, `images`, `videos`, `store`, `categories`, `details`, `parent_asin`, `bought_together`, `subtitle`, `author`

**Surprises**
- `subtitle` and `author` are **not in the official documentation**. They look like fields from Amazon **book** pages. They're probably empty for normal car parts and filled for books or repair manuals. *To check in notebook 02.*
- `timestamp` is a big whole number (BIGINT): milliseconds since 1 Jan 1970, not a readable date yet.
- Several columns pack lists or records into one cell (`features`, `description`, `categories`, `images`, `details`).

Every column is explained in [docs/03-data-dictionary.md](../docs/03-data-dictionary.md).

**Next:** notebook 02 explores the values: blanks, fake blanks, duplicates, odd values and dates.